# PCB test-point placement — DreamerV3 cold-start training (A100, static canonical board)

Trains the placement policy on the **real TE AutoLayout Example01 board** (`--boards canonical`: the exact 135×90mm board from `AutoLayout_Example01.xlsx`, **static — the same geometry every episode**) using the full cold-start stack — expert demos, anchored behavior cloning (10% floor), potential reward shaping, single-layer reward, and the exact-geometry vector observation. This is single-instance optimization: the policy's job is to place test points on *this* board better than the classical pipeline, and the scoreboard at the end measures exactly that.

**Setup:** `Runtime → Change runtime type → A100 GPU` (Colab Pro), then run all cells top to bottom. The Drive cell asks for authorization once.

Timeline: demo generation ~2 min, then 60k training steps ≈ 2–4 h with checkpoints every 5k steps. Interrupt anytime: re-running resumes exactly where it left off.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100 GPU"
name = torch.cuda.get_device_name(0)
print("torch", torch.__version__, "|", name)
if "A100" not in name:
    print("NOTE: not an A100 — everything still runs, just slower; "
          "consider CONFIG='colab', NUM_TRACES=8 in the settings cell.")

In [ ]:
# All logs, demos, and checkpoints live in your Google Drive, so nothing is
# lost on disconnect and re-running this notebook later resumes training.
from google.colab import drive
drive.mount("/content/drive")
LOGROOT = "/content/drive/MyDrive/pcb-router-logs"
print("backing up to:", LOGROOT)

In [ ]:
# Get the code from GitHub main (pulls the latest on re-runs).
import pathlib
if not pathlib.Path("/content/pcb-router").exists():
    !git clone -q https://github.com/pauljiang03/pcb-router /content/pcb-router
%cd /content/pcb-router
!git pull --ff-only

In [ ]:
# Dependencies (torch/numpy/tensorboard/matplotlib ship with Colab) and a
# quick sanity run of the cold-start tests (~10 s).
%pip -q install gymnasium "ruamel.yaml" openpyxl
!python -m pytest tests/test_coldstart.py -q

In [ ]:
NUM_TRACES = 20        # the canonical board has exactly 20 traces
CONFIG = "colab_a100"  # configs.yaml section (T4 fallback: "colab")
# "canonical" = the REAL TE AutoLayout Example01 board (xlsx, 135x90mm),
# STATIC: the exact scoreboard geometry every single episode. This is
# single-instance optimization -- learning the best sequence for THIS board
# is the deliverable. ("canonical_family" = the old jittered variant.)
BOARDS = "canonical"
STEPS = 60000
# smart_placement is deterministic on a fixed board, so demos are identical
# by design -- a couple dozen suffice as the BC anchor (~2 min to generate).
DEMOS = 25
# 4 env workers overlap CPU routing with GPU training (A100 VMs have 12 vCPUs).
ENV_FLAGS = "--envs 4 --parallel"
# eval.py must score the same board the run trains on:
EVAL_FLAGS = "--board canonical" if "canonical" in BOARDS else ""
# Fresh dir for the static run (never reuse dirs from jittered runs).
RUN_DIR = f"{LOGROOT}/canonical-static"
print(NUM_TRACES, "traces |", CONFIG, "|", BOARDS, "|", STEPS, "steps |", RUN_DIR)

In [ ]:
# Live training curves. Key scalars:
#   log_routable  -- the live planarity metric: +10 = all traces routed on ONE
#                    layer, -5 per planar failure; want the average -> +10
#   bc_loss       -- imitation fit; should fall toward ~1 and never vanish
#                    (bc_floor keeps a permanent 10% anchor)
#   imag_reward_mean -- exploitation alarm: real per-step reward can't exceed
#                    ~+2; if this balloons, the model is hallucinating again
#   eval_return   -- compare against the demo running-mean printed during
#                    demo generation (that number IS the baseline anchor)
%load_ext tensorboard
import tensorboard.notebook as tbnb
tbnb.start("--logdir " + LOGROOT)

## Train — static canonical board, cold start (demos + anchored BC + shaping)

First run generates 25 expert episodes into `demo_eps/` (~2 min; they're identical by design — `smart_placement` is deterministic on a fixed board — and serve as the BC anchor: the classical 20/20, max-95mm solution). **The anchor line it prints is the return eval must reach; beating it = beating the classical pipeline.** Then it trains 60k steps on the exact board, every episode. Interrupt anytime; re-run this cell to resume.

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{RUN_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --boards {BOARDS} --steps {STEPS} --demos {DEMOS}

In [ ]:
# Score the trained policy against the classical baselines on the SAME
# canonical board (the summary table is the headline result: compare the
# Dreamer row to Smart on failures / max / spread). Add --board_seed 1000000
# for held-out jittered variants; --fast for a quick low-budget pass.
!python eval.py --checkpoint "{RUN_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot {EVAL_FLAGS}

In [ ]:
# Render the routed boards to PNGs (~2-3 min: same quality router as the
# table above, one episode per method), save them to Drive, and display the
# headline comparison inline: Smart (classical) vs Dreamer (learned).
!python eval.py --checkpoint "{RUN_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 1 --num_traces {NUM_TRACES} --device cuda:0 {EVAL_FLAGS}

import pathlib, shutil
from IPython.display import Image, display
figs = pathlib.Path(RUN_DIR) / "figs"
figs.mkdir(exist_ok=True)
for p in sorted(pathlib.Path("eval_results").glob("*_1.png")):
    shutil.copy(p, figs / p.name)
print("figures saved to", figs)
for name in ("smart_1.png", "dreamer_1.png"):
    p = pathlib.Path("eval_results") / name
    if p.exists():
        print("\n===", name, "===")
        display(Image(str(p)))

## Beat the baseline: search → distill (expert iteration)

The policy clones its demos; on a static board it plateaus *at* them. So let search find a better solution first: local search with the router in the loop (a 200s probe already took the classical 95mm/1332mm solution to **86mm/1073mm**, still improving). Then distill: regenerate the demos from the optimized placement and resume training anchored to the *new* target — the policy learns to reproduce the search result one-shot, and RL polishes from there. Eval now also runs the serpentine equalization post-stage (`matched=n/n`, `eq_spread`) and the PNGs render the as-built equalized board.

In [ ]:
# 1) Search: ~15 CPU-min of router-in-the-loop local search from the smart
# solution. Verifies routing AND equalization (matched=20/20) at the end,
# saves the best placement to Drive.
!python scripts/optimize_placement.py --board canonical --minutes 15 \
    --out "{RUN_DIR}/best_placement.json"

In [ ]:
# 2) Distill: replace the demos with the optimized placement and resume.
# --bc_scale 10 x the 0.1 floor = effective FULL-strength anchor to the new
# target even though the decay schedule has run out; --steps 90000 extends
# the finished run by ~30k steps. NOTE: at full anchor the policy CLONES the
# search result and freezes there -- that is the goal (one-shot reproduction
# of the best-known board). There is no RL-past-the-anchor in this phase; if
# you ever want polish, run a further extension with --bc_scale 1 (weak 0.1
# anchor) -- but search already sits at a local optimum, so expect little.
!rm -f "{RUN_DIR}"/demo_eps/*.npz
!python train.py --configs defaults {CONFIG} --logdir "{RUN_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --boards {BOARDS} --steps 90000 --demos {DEMOS} \
    --demo_placement "{RUN_DIR}/best_placement.json" --bc_scale 10
# Then re-run the scoreboard + PNG cells above: the bar is failures=0,
# max at the optimizer's number, matched=20/20.

## World-model-guided search (the surrogate experiment)

On a static board, search beats the policy — but the **world model** still has a distinctive job: every observation component is pure geometry (routing only ever enters the terminal reward), so the full episode for any candidate placement can be built *without routing* and scored by the reward head in one batched forward. That makes the trained world model a **millisecond router surrogate**: score a batch of mutations, spend real router calls only on the top-k. `wm_guided_search.py` measures the two numbers that matter — **surrogate fidelity** (Spearman ρ between predicted and true returns, plus ρ against −max on the routable subset) and **guided vs blind search at equal router calls** — and saves a distillable placement JSON.

In [ ]:
# Surrogate search: the world model scores placements WITHOUT the router
# (observations are pure geometry, so the reward head's summed prediction on
# a built episode is a predicted return). Measures (1) surrogate fidelity --
# rank correlation between predicted and true returns -- and (2) guided vs
# blind search at equal router calls (blind continues to 3x C to measure the
# headroom, so expect the blind phase to take ~2-3x the guided minutes).
# Point CKPT at whichever static-run checkpoint exists: the completed 60k
# static run lives in the versioned v4 Drive dir; a run trained by THIS
# notebook is at RUN_DIR/latest.pt.
CKPT = "/content/drive/MyDrive/pcb-router-logs-v4/canonical/latest.pt"
# CKPT = f"{RUN_DIR}/latest.pt"
!python scripts/wm_guided_search.py --checkpoint "{CKPT}" \
    --configs defaults colab_a100 --minutes 10 --fidelity 32 \
    --device cuda:0 --out "{RUN_DIR}/wm_search.json"
# Reading it: fidelity rho >= 0.6 -> the surrogate genuinely ranks
# placements; 0.3-0.6 partial guidance; < 0.3 the reward head does not rank
# off-policy mutations (honest extension: fine-tune it on search trial logs).
# The speedup claim holds if "blind calls to match guided" >= 3x guided's C.
# The JSON is --demo_placement compatible, so a guided win distills exactly
# like the blind optimizer's (cell above).

## Reading the results

- **`log_routable`** — the planarity metric. +10 = all 20 traces routed on the single copper layer; each planar failure costs −5. The demos sit at +10; the policy's average should climb there and stay.
- **`bc_loss`** — imitation fit. Starts ≈5 (uniform over 200 candidates); with the vector observation it should fall steadily toward ~1. It never disappears — `bc_floor` keeps a permanent 10% anchor precisely so the policy can't drift into states where the world model hallucinates.
- **`imag_reward_mean`** — the exploitation alarm. Real per-step reward tops out around +2; if this balloons to +10 or more, the reward head is hallucinating again and the run is suspect.
- **`eval_return` vs the demo anchor** — the demo generator's running-mean return is the baseline. Reaching it = the cold start worked; exceeding it = RL is finding shorter, better-matched placements than the heuristic (watch `log_length_max` and `log_spread` shrink toward 0 — that's the real objective on this already-routable board).
- **`eval.py` summary table** — the scoreboard vs. classical baselines. Beating Smart on `max=` / `spread=` on the TE board is the win condition for the single-board model. Watch that failures never rise while return improves.

Everything is already backed up in your Drive — stopping the runtime loses nothing.

In [ ]:
# Optional: download a local copy of the checkpoint + TensorBoard events.
# (Your Drive already has everything.)
import pathlib, shutil
out = pathlib.Path("/content/results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
d = pathlib.Path(RUN_DIR)
for f in list(d.glob("events*")) + [d / "latest.pt"]:
    if f.exists():
        shutil.copy(f, out / f.name)
shutil.make_archive("/content/pcb_router_results", "zip", out)
from google.colab import files
files.download("/content/pcb_router_results.zip")